# 1. Setup

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import html
import random

from google.colab import userdata
import jax
import keras
from keras import layers
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [ ]:
DATASET_PATH = userdata.get(DATASET_PATH)
TRAIN_CSV = DATASET_PATH + 'train.csv'
TEST_CSV = DATASET_PATH + 'test.csv'

TRAINING_SEEDS = [123, 231, 321]
CORRUPTION_SEED = 0
SPLIT_SEED = 1

CORRUPTIONS = ['swap', 'substitution', 'deletion', 'insertion']
CORRUPTION_LEVELS = [0, 5, 10, 15, 20]

STANDARDIZATION = 'lower_and_strip_punctuation'

WORD_MAX_TOKENS = 5000
WORD_SEQUENCE_LENGTH = 69
WORD_EMBEDDING_DIM = 32

CHAR_MAX_TOKENS = 200
CHAR_SEQUENCE_LENGTH = 443     
CHAR_EMBEDDING_DIM = 32

CONV_FILTERS = 128
KERNEL_SIZE = 5
DENSE_UNITS = 64

HIDDEN_ACTIVATION = "relu"
OUTPUT_ACTIVATION = "softmax"

DROPOUT_RATE = 0.2

NUM_CLASSES = 4

LEARNING_RATE = 1e-3
ALPHA = 0.01
DECAY_STEPS = 2820

EPOCHS = 50
PATIENCE = 5
BATCH_SIZE = 64

In [24]:
QWERTY_NEIGHBORS = {
    '`': ['1', 'q'],
    '~': ['!', 'Q'],
    '1': ['`', '2', 'q', 'w'],
    '!': ['~', '@', 'Q', 'W'],
    '2': ['1', '3', 'w', 'e'],
    '@': ['!', '#', 'W', 'E'],
    '3': ['2', '4', 'e', 'r'],
    '#': ['@', '$', 'E', 'R'],
    '4': ['3', '5', 'r', 't'],
    '$': ['#', '%', 'R', 'T'],
    '5': ['4', '6', 't', 'y'],
    '%': ['$', '^', 'T', 'Y'],
    '6': ['5', '7', 'y', 'u'],
    '^': ['%', '&', 'Y', 'U'],
    '7': ['6', '8', 'u', 'i'],
    '&': ['^', '*', 'U', 'I'],
    '8': ['7', '9', 'i', 'o'],
    '*': ['&', '(', 'I', 'O'],
    '9': ['8', '0', 'o', 'p'],
    '(': ['*', ')', 'O', 'P'],
    '0': ['9', '-', 'p', '['],
    ')': ['(', '_', 'P', '{'],
    '-': ['0', '=', '[', ']'],
    '_': [')', '+', '{', '}'],
    '=': ['-', ']', '['],
    '+': ['_', '}', '|'],

    'q': ['`', '1', 'w', 'a'],
    'Q': ['~', '!', 'W', 'A'],
    'w': ['1', '2', 'q', 'e', 'a', 's'],
    'W': ['!', '@', 'Q', 'E', 'A', 'S'],
    'e': ['2', '3', 'w', 'r', 's', 'd'],
    'E': ['@', '#', 'W', 'R', 'S', 'D'],
    'r': ['3', '4', 'e', 't', 'd', 'f'],
    'R': ['#', '$', 'E', 'T', 'D', 'F'],
    't': ['4', '5', 'r', 'y', 'f', 'g'],
    'T': ['$', '%', 'R', 'Y', 'F', 'G'],
    'y': ['5', '6', 't', 'u', 'g', 'h'],
    'Y': ['%', '^', 'T', 'U', 'G', 'H'],
    'u': ['6', '7', 'y', 'i', 'h', 'j'],
    'U': ['^', '&', 'Y', 'I', 'H', 'J'],
    'i': ['7', '8', 'u', 'o', 'j', 'k'],
    'I': ['&', '*', 'U', 'O', 'J', 'K'],
    'o': ['8', '9', 'i', 'p', 'k', 'l'],
    'O': ['*', '(', 'I', 'P', 'K', 'L'],
    'p': ['9', '0', 'o', '[', 'l', ';'],
    'P': ['(', ')', 'O', '{', 'L', ':'],
    '[': ['0', '-', 'p', ']', ';', "'"],
    '{': [')', '_', 'P', '}', ':', '"'],
    ']': ['-', '=', '[', '\\', "'"],
    '}': ['_', '+', '{', '|', '"'],
    '\\': ['=', ']'],
    '|': ['+', '}'],

    'a': ['q', 'w', 's', 'z'],
    'A': ['Q', 'W', 'S', 'Z'],
    's': ['w', 'e', 'a', 'd', 'z', 'x'],
    'S': ['W', 'E', 'A', 'D', 'Z', 'X'],
    'd': ['e', 'r', 's', 'f', 'x', 'c'],
    'D': ['E', 'R', 'S', 'F', 'X', 'C'],
    'f': ['r', 't', 'd', 'g', 'c', 'v'],
    'F': ['R', 'T', 'D', 'G', 'C', 'V'],
    'g': ['t', 'y', 'f', 'h', 'v', 'b'],
    'G': ['T', 'Y', 'F', 'H', 'V', 'B'],
    'h': ['y', 'u', 'g', 'j', 'b', 'n'],
    'H': ['Y', 'U', 'G', 'J', 'B', 'N'],
    'j': ['u', 'i', 'h', 'k', 'n', 'm'],
    'J': ['U', 'I', 'H', 'K', 'N', 'M'],
    'k': ['i', 'o', 'j', 'l', 'm', ','],
    'K': ['I', 'O', 'J', 'L', 'M', '<'],
    'l': ['o', 'p', 'k', ';', ',', '.'],
    'L': ['O', 'P', 'K', ':', '<', '>'],
    ';': ['p', '[', 'l', "'", '.', '/'],
    ':': ['P', '{', 'L', '"', '>', '?'],
    "'": ['[', ']', ';', '/'],
    '"': ['{', '}', ':', '?'],

    'z': ['a', 's', 'x'],
    'Z': ['A', 'S', 'X'],
    'x': ['s', 'd', 'z', 'c'],
    'X': ['S', 'D', 'Z', 'C'],
    'c': ['d', 'f', 'x', 'v'],
    'C': ['D', 'F', 'X', 'V'],
    'v': ['f', 'g', 'c', 'b'],
    'V': ['F', 'G', 'C', 'B'],
    'b': ['g', 'h', 'v', 'n'],
    'B': ['G', 'H', 'V', 'N'],
    'n': ['h', 'j', 'b', 'm'],
    'N': ['H', 'J', 'B', 'M'],
    'm': ['j', 'k', 'n', ','],
    'M': ['J', 'K', 'N', '<'],
    ',': ['k', 'l', 'm', '.'],
    '<': ['K', 'L', 'M', '>'],
    '.': ['l', ';', ',', '/'],
    '>': ['L', ':', '<', '?'],
    '/': [';', "'", '.'],
    '?': [':', '"', '>']
}

In [25]:
tf.config.experimental.enable_op_determinism()
corruption_rng = np.random.default_rng(CORRUPTION_SEED)

In [26]:
jax.devices('cpu')

[CpuDevice(id=0)]

In [27]:
try:
    print(jax.devices('gpu'))
except:
    print('No GPU')

No GPU


# 2. Data

## 2.1 Loading

In [28]:
train_val_set = pd.read_csv(TRAIN_CSV).groupby('Class Index', sort=False).head(3500)
train_val_set = train_val_set.drop(columns=['Title'])
train_val_set['Description'] = train_val_set['Description'].map(html.unescape)
train_val_set['Description'] = train_val_set['Description'].str.replace('\\', ' ').str.replace('quot;', '').str.replace('--', '-').str.replace(r'\s+', ' ', regex=True).str.strip()
train_val_set

,Class Index,Description
0,3,"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Reuters - Private investment firm Carlyle Grou...
2,3,Reuters - Soaring crude prices plus worries ab...
3,3,Reuters - Authorities have halted oil export f...
4,3,"AFP - Tearaway world oil prices, toppling reco..."
...,...,...
14654,3,No. 7 carrier must reach agreements with union...
14655,3,"China Petroleum amp; Chemical Corp., Asia #39;..."
14661,3,Reuters - U.S. consumer spending rebounded sha...
14662,3,Reuters - U.S. stocks are set to open lower on...


In [29]:
p99_chars = train_val_set['Description'].str.len().quantile(0.99)

p99_words = train_val_set['Description'].str.split().str.len().quantile(0.99)

print('99th percentile characters:', p99_chars)
print('99th percentile words:', p99_words)

99th percentile characters: 443.0
99th percentile words: 69.0


In [30]:
train_df: pd.DataFrame
val_df: pd.DataFrame 

train_df, val_df = train_test_split(train_val_set, test_size=1/7, random_state=SPLIT_SEED, stratify=train_val_set['Class Index'])
val_df['Class Index'].value_counts()

,count
Class Index,
4,500
2,500
1,500
3,500


In [31]:
train_df['Class Index'].value_counts()

,count
Class Index,
2,3000
4,3000
1,3000
3,3000


In [32]:
test_df = pd.read_csv(TRAIN_CSV).groupby("Class Index", sort=False).head(500).drop(columns=['Title'])
test_df['Description'] = test_df['Description'].map(html.unescape)
test_df['Description'] = test_df['Description'].str.replace('\\', '').str.replace('quot;', '').str.replace('--', '-').str.replace(r'\s+', ' ', regex=True).str.strip()
test_df['Class Index'].value_counts()

,count
Class Index,
3,500
4,500
2,500
1,500


In [33]:
test_df

,Class Index,Description
0,3,"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Reuters - Private investment firm Carlyle Grou...
2,3,Reuters - Soaring crude prices plus worriesabo...
3,3,Reuters - Authorities have halted oil exportfl...
4,3,"AFP - Tearaway world oil prices, toppling reco..."
...,...,...
2488,2,"What if the Olympic Games -started here 2,780 ..."
2489,2,More security will be placed around the fields...
2498,2,After an hour of interrogation by a three-man ...
2506,2,"ATHENS, Greece - The night before, Michael Phe..."


## 2.2 Corruption

In [34]:
def apply_corruption(row: str, corruption_level: int) -> str:
    corruption_chance = corruption_level / 100
    words = row.split(' ')
    for k, word in enumerate(words):
        if corruption_rng.random() < corruption_chance:
            chars = list(word)
            i = corruption_rng.integers(len(chars))
            match corruption_rng.choice(CORRUPTIONS):
                case 'swap':
                    if len(chars) > 1:
                        if i < len(chars) - 1 and chars[i+1] != chars[i]:
                            chars[i], chars[i + 1] = chars[i + 1], chars[i]
                        else:
                            chars[i], chars[i - 1] = chars[i - 1], chars[i]
                case 'substitution':
                    chars[i] = corruption_rng.choice(QWERTY_NEIGHBORS[chars[i]])
                case 'deletion':
                    del chars[i]
                case 'insertion':
                    chars.insert(i+1, corruption_rng.choice(QWERTY_NEIGHBORS[chars[i]]))
            words[k] = ''.join(chars)
    return ' '.join(words)

In [35]:
test_sets: dict[int, pd.DataFrame] = {}

for corruption_level in CORRUPTION_LEVELS:
    test_sets[corruption_level] = test_df.assign(Description=test_df['Description'].map(lambda x: apply_corruption(x, corruption_level))) #type: ignore

In [36]:
test_sets

{0:       Class Index                                        Description
 0               3  Reuters - Short-sellers, Wall Street's dwindli...
 1               3  Reuters - Private investment firm Carlyle Grou...
 2               3  Reuters - Soaring crude prices plus worriesabo...
 3               3  Reuters - Authorities have halted oil exportfl...
 4               3  AFP - Tearaway world oil prices, toppling reco...
 ...           ...                                                ...
 2488            2  What if the Olympic Games -started here 2,780 ...
 2489            2  More security will be placed around the fields...
 2498            2  After an hour of interrogation by a three-man ...
 2506            2  ATHENS, Greece - The night before, Michael Phe...
 2507            2  ATHENS, Greece The slogan for these Olympics i...
 
 [2000 rows x 2 columns],
 5:       Class Index                                        Description
 0               3  Reuters - Short-sellers, Wall Street

## 2.3 Vectorization

In [37]:
word_vectorizer = keras.layers.TextVectorization(max_tokens=WORD_MAX_TOKENS, standardize=STANDARDIZATION, split='whitespace', output_mode='int', output_sequence_length=WORD_SEQUENCE_LENGTH)
word_vectorizer.adapt(train_df['Description'])

char_vectorizer = keras.layers.TextVectorization(max_tokens=CHAR_MAX_TOKENS, standardize=STANDARDIZATION, split='character', output_mode='int', output_sequence_length=CHAR_SEQUENCE_LENGTH)
char_vectorizer.adapt(train_df['Description'])

In [38]:
words_train_X = word_vectorizer(train_df['Description'].to_numpy())
words_train_Y = train_df['Class Index'] - 1

words_val_X = word_vectorizer(train_df['Description'].to_numpy())
words_val_Y = train_df['Class Index'] - 1

In [39]:
char_train_X = char_vectorizer(train_df['Description'].to_numpy())
char_train_Y = train_df['Class Index'] - 1

char_val_X = char_vectorizer(train_df['Description'].to_numpy())
char_val_Y = train_df['Class Index'] - 1

# 3. Models

In [40]:
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=DECAY_STEPS,
    alpha=ALPHA,
)

optimizer=keras.optimizers.Adam(
    learning_rate=lr_schedule
)

In [48]:
word_model = keras.Sequential([
    keras.Input(
        shape=(WORD_SEQUENCE_LENGTH,),
        batch_size=BATCH_SIZE,
        dtype='int32'
    ),

    layers.Embedding(
        input_dim=len(word_vectorizer.get_vocabulary()),
        output_dim=WORD_EMBEDDING_DIM,
    ),

    layers.Conv1D(
        filters=CONV_FILTERS,
        kernel_size=KERNEL_SIZE,
        activation='relu',
    ),

    layers.GlobalMaxPooling1D(),

    layers.Dense(
        units=DENSE_UNITS,
        activation='relu',
    ),

    layers.Dense(
        units=NUM_CLASSES,
        activation='softmax',
    ),
])

word_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

word_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (64, 69, 32)           │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (64, 65, 128)          │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_6          │ (64, 128)              │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (64, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (64, 4)                │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 189,124 (738.77 KB)

 Trainable params: 189,124 (738.77 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
char_model = keras.Sequential([
    keras.Input(
        shape=(CHAR_SEQUENCE_LENGTH,),
        batch_size=BATCH_SIZE,
        dtype='int32'
    ),
    
    layers.Embedding(
        input_dim=len(char_vectorizer.get_vocabulary()),
        output_dim=CHAR_EMBEDDING_DIM,
    ),

    layers.Conv1D(
        filters=CONV_FILTERS,
        kernel_size=KERNEL_SIZE,
        activation='relu',
    ),

    layers.GlobalMaxPooling1D(),

    layers.Dense(
        units=DENSE_UNITS,
        activation='relu',
    ),

    layers.Dense(
        units=NUM_CLASSES,
        activation='softmax',
    ),
])

char_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

char_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (64, 443, 32)          │         1,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (64, 439, 128)         │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_8          │ (64, 128)              │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (64, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (64, 4)                │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,372 (118.64 KB)

 Trainable params: 30,372 (118.64 KB)

 Non-trainable params: 0 (0.00 B)

In [55]:
print(f'Word model has {(word_model.count_params()/char_model.count_params()):.2f}x as many parameters than the character model')

Word model has 6.23x as many parameters than the character model


# 4. Training